# Smart Bin — Provenance audit (Google Colab)

**Why this exists:** the paper's dataset folder holds **6,851 images** (Paper 1,831 / Plastic 1,674 / Food 1,461 / Others 1,885), but the published RealWaste dataset is only **4,752** images, and the 284 phone photos you added explain only part of the gap. **1,815 images are still unaccounted for.** A reviewer who checks this will ask where they came from — augmentation, duplicates, or a second source. Run this audit *before* submitting to ICECT 2027 (deadline **10 Nov 2026**) so the paper can state the answer honestly instead of leaving the gap open.

**What it does, in order:**
1. Finds `realwaste1.zip` in Drive and unzips it (reuses `/content/realwaste_unzipped` if already there).
2. Counts exact duplicates by MD5 file hash, per class.
3. Flags near-duplicates (same size + dimensions, and perceptual-hash distance ≤ 5 where `imagehash` installs).
4. Checks whether identical images leaked across the train / val / test split folders (the split was image-level, so the same item's photos can sit in both train and test).
5. Writes `provenance_audit.json` + `provenance_audit.txt` to Drive so the numbers can go straight into the paper.

It changes nothing in your dataset — read-only except the two report files it writes.

In [ ]:
# Cell 1 — Mount Drive, find and unzip realwaste1.zip (skip if already unzipped)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

ZIP_NAME = "realwaste1.zip"
UNZIP_DIR = Path("/content/realwaste_unzipped")

zips = list(Path("/content/drive/MyDrive").rglob(ZIP_NAME))
assert zips, f"{ZIP_NAME} not found anywhere in MyDrive"
zip_path = zips[0]
print("zip:", zip_path)

if not (UNZIP_DIR.exists() and any(UNZIP_DIR.iterdir())):
    import zipfile
    print("unzipping ...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(UNZIP_DIR)
    print("done.")
else:
    print("already unzipped, reusing:", UNZIP_DIR)

# Locate the folder holding the 4 class folders (zip contains an 'archive' folder)
def find_class_root(base):
    for p in base.rglob("*"):
        if p.is_dir() and {d.name for d in p.iterdir() if d.is_dir()} >= {"Paper", "Plastic", "Food", "Others"}:
            return p
    return None

CLASS_ROOT = find_class_root(UNZIP_DIR)
assert CLASS_ROOT is not None, "Could not find a folder containing Paper/Plastic/Food/Others"
print("class root:", CLASS_ROOT)
for cls in ["Paper", "Plastic", "Food", "Others"]:
    n = len(list((CLASS_ROOT / cls).glob("*")))
    print(f"  {cls}: {n} files on disk")


In [ ]:
# Cell 2 — Exact duplicates (MD5) + near-duplicate candidates, per class
import hashlib, json
from collections import Counter, defaultdict
from pathlib import Path

CLASSES = ["Paper", "Plastic", "Food", "Others"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

report = {}
for cls in CLASSES:
    files = sorted(p for p in (CLASS_ROOT / cls).iterdir()
                   if p.is_file() and p.suffix.lower() in IMG_EXTS)
    hashes = {}
    for p in files:
        hashes.setdefault(md5_of(p), []).append(str(p.name))
    dup_groups = {h: names for h, names in hashes.items() if len(names) > 1}
    extra_copies = sum(len(v) - 1 for v in dup_groups.values())
    report[cls] = {
        "total_files": len(files),
        "unique_md5": len(hashes),
        "exact_duplicate_groups": len(dup_groups),
        "extra_copies_beyond_first": extra_copies,
        "unique_after_exact_dedup": len(files) - extra_copies,
    }
    print(f"{cls}: {len(files)} files, {len(hashes)} unique, "
          f"{len(dup_groups)} duplicate groups, {extra_copies} extra copies")

total = sum(r["total_files"] for r in report.values())
unique = sum(r["unique_after_exact_dedup"] for r in report.values())
print(f"\nTOTAL: {total} files -> {unique} unique after exact dedup "
      f"({total - unique} extra copies)")
print("Published RealWaste total is 4,752; phone photos added were 284; "
      "so 6,851 - 4,752 - 284 = 1,815 still unexplained BEFORE dedup.")
print(f"After removing exact duplicates, unexplained = {unique} - 4,752 - 284 = {unique - 4752 - 284}")


In [ ]:
# Cell 3 — Near-duplicates: same dimensions + file size, plus perceptual hash if available
# (Exact MD5 misses re-saved / re-compressed copies of the same photo.)
from collections import defaultdict
from pathlib import Path
from PIL import Image

try:
    import imagehash
    HAVE_IHASH = True
except ImportError:
    HAVE_IHASH = False
    print("imagehash not installed — run: pip install imagehash  (then re-run this cell)")
    print("Falling back to (width, height, file-size) grouping only.")

for cls in CLASSES:
    files = sorted(p for p in (CLASS_ROOT / cls).iterdir()
                   if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"})
    sig_groups = defaultdict(list)
    for p in files:
        try:
            with Image.open(p) as im:
                sig_groups[(im.size, p.stat().st_size)].append(p.name)
        except Exception as e:
            print(f"  unreadable: {p.name} ({e})")
    cand = {k: v for k, v in sig_groups.items() if len(v) > 1}
    print(f"{cls}: {len(cand)} same-size+dimension groups "
          f"({sum(len(v)-1 for v in cand.values())} extra files) — inspect these first")

    if HAVE_IHASH:
        hashes = {}
        for p in files:
            try:
                with Image.open(p) as im:
                    hashes[p.name] = imagehash.phash(im)
            except Exception:
                pass
        names = list(hashes)
        near = 0
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                if hashes[names[i]] - hashes[names[j]] <= 5:
                    near += 1
        print(f"{cls}: {near} perceptual-hash pairs within distance 5 (possible re-saves/crops)")


In [ ]:
# Cell 4 — Train/val/test leakage: same image in more than one split?
# Only runs if split folders exist (created by the retraining notebook).
import hashlib
from pathlib import Path
from collections import defaultdict

SPLIT_ROOTS = [Path("/content/split"), Path("/content/realwaste_split"),
               CLASS_ROOT.parent / "split"]
split_root = next((p for p in SPLIT_ROOTS if p.exists()), None)
if split_root is None:
    print("No split folders found — leakage check skipped.")
    print("If you ran the retraining notebook, its train/val/test folders live under "
          "/content/smartbin_runs or similar; point SPLIT_ROOTS at them and re-run.")
else:
    print("split root:", split_root)
    where = defaultdict(set)  # md5 -> {train, val, test}
    for split in ["train", "val", "test"]:
        for cls in CLASSES:
            d = split_root / split / cls
            if not d.exists():
                continue
            for p in d.iterdir():
                if p.is_file():
                    h = hashlib.md5(p.read_bytes()).hexdigest()
                    where[h].add(split)
    leaked = {h: s for h, s in where.items() if len(s) > 1}
    print(f"images appearing in >1 split: {len(leaked)} of {len(where)} unique")
    if leaked:
        print("-> The split is image-level, not item-level: report this as a limitation, "
              "or re-split by image identity before any re-run.")
    else:
        print("-> No identical image sits in two splits (note: near-identical photos of the "
              "same item can still leak — only exact hashes are checked here).")


In [ ]:
# Cell 5 — Write the report to Drive (send these two files back for the paper)
import json, datetime
from pathlib import Path

out_dir = Path("/content/drive/MyDrive/SmartBin_Results") / f"provenance_audit_{datetime.date.today():%Y%m%d}"
out_dir.mkdir(parents=True, exist_ok=True)

summary = {
    "date": str(datetime.date.today()),
    "class_root": str(CLASS_ROOT),
    "per_class_exact_dedup": report,
    "totals": {
        "files_on_disk": total,
        "unique_after_exact_dedup": unique,
        "published_realwaste_total": 4752,
        "phone_photos_added": 284,
        "unexplained_before_dedup": 6851 - 4752 - 284,
        "unexplained_after_exact_dedup": unique - 4752 - 284,
    },
    "note": ("Unexplained images are NOT labelled as duplicates or augmentation until "
             "Cells 2-3 are inspected. If unexplained_after_exact_dedup is still large, "
             "the paper must say so honestly, or find the second source."),
}
(out_dir / "provenance_audit.json").write_text(json.dumps(summary, indent=2))
(out_dir / "provenance_audit.txt").write_text(
    "Smart Bin provenance audit\n" + json.dumps(summary, indent=2) + "\n")
print("wrote:", out_dir / "provenance_audit.json")
print("wrote:", out_dir / "provenance_audit.txt")
print("\nSend back: provenance_audit.json + the duplicate-group counts printed above.")
print("What to send back for the paper: per-class unique counts, total extra copies, "
      "and whether any train/val/test leakage was found.")


## What to send back (for the paper)

From the Drive folder, send `provenance_audit.json` plus the printed counts:
- per-class unique images after exact dedup,
- total extra copies (exact duplicates),
- near-duplicate groups worth a look,
- train/val/test leakage: yes / no, how many.

**Do not** write the audit's conclusion into the paper in advance. If the unexplained count drops to ~0 after dedup, the paper can say the extra images were duplicates. If it stays large, the paper must keep the gap flagged as a limitation — or the second source must be found before submission.